In [1]:
from category_encoders import TargetEncoder
import numpy as np
import os
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
train_df = pd.read_csv(os.path.join('..', 'data', 'processed', 'train_cleaned.csv'))
test_df = pd.read_csv(os.path.join('..', 'data', 'processed', 'test_cleaned.csv'))

print(f"Train date range: {train_df['Year'].min()} to {train_df['Year'].max()}")
print(f"Train date range: {test_df['Year'].min()} to {test_df['Year'].max()}")

Train date range: 1990 to 2020
Train date range: 2021 to 2024


In [3]:
# Shares & Ratios
def add_core_features(df):    
    # Fossil vs clean shares
    df["fossil_share"] = (
        df["Electricity production from coal sources (% of total)"].fillna(0) +
        df["Electricity production from oil sources (% of total)"].fillna(0) +
        df["Electricity production from natural gas sources (% of total)"].fillna(0)
    )
    
    df["clean_share"] = (
        df["Electricity production from hydroelectric sources (% of total)"].fillna(0) +
        df["Electricity production from nuclear sources (% of total)"].fillna(0) +
        df["Electricity production from renewable sources, excluding hydroelectric (% of total)"].fillna(0)
    )
    
    # Energy intensity style features
    df["energy_x_fossil"] = df["Energy use (kg of oil equivalent per capita)"] * df["fossil_share"]
    df["energy_x_clean"] = df["Energy use (kg of oil equivalent per capita)"] * df["clean_share"]
    
    # Urbanization intensity
    df["urban_energy"] = df["Urban population (% of total population)"] * df["Energy use (kg of oil equivalent per capita)"]
    
    return df

train_df = add_core_features(train_df)
test_df = add_core_features(test_df)

In [4]:
# Per‑Capita Conversions
def add_per_capita(df):
    pop = df["Population, total"]
    
    df["ag_land_sqkm_per_capita"] = df["Agricultural land (sq. km)"] / pop
    df["forest_area_sqkm_per_capita"] = df["Forest area (sq. km)"] / pop
    df["water_withdrawal_bcm_per_capita"] = df["Annual freshwater withdrawals, total (billion cubic meters)"] / pop
    
    return df

train_df = add_per_capita(train_df)
test_df = add_per_capita(test_df)

In [5]:
# Log Transforms
def add_log_features(df):
    cols = [
        "Population, total",
        "Urban population",
        "Energy use (kg of oil equivalent per capita)",
        "Electric power consumption (kWh per capita)",
        "Agricultural land (sq. km)",
        "Forest area (sq. km)",
        "Electricity production from renewable sources, excluding hydroelectric (kWh)"
    ]
    
    for col in cols:
        if col in df.columns:
            df[f"log1p_{col}"] = np.log1p(df[col])
    
    return df

train_df = add_log_features(train_df)
test_df = add_log_features(test_df)

/Users/benjaminkwan/coding_projects/carbon_dioxide_emissions/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/benjaminkwan/coding_projects/carbon_dioxide_emissions/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [6]:
# Growth Rates (YoY %)
def add_pct_change(df):
    cols = [
        "Energy use (kg of oil equivalent per capita)",
        "Electric power consumption (kWh per capita)",
        "Population, total",
        "Urban population",
        "fossil_share"
    ]

    g = df.groupby("Country Name", sort=False)
    
    for col in cols:
        df[f"{col}_yoy_pct"] = g[col].pct_change()

    return df

train_df = add_pct_change(train_df)
test_df = add_pct_change(test_df)

In [7]:
# Rolling Windows (3‑year mean/std)
def add_rolling(df, window=3):
    cols = [
        "Energy use (kg of oil equivalent per capita)",
        "Electric power consumption (kWh per capita)",
        "fossil_share"
    ]

    g = df.groupby("Country Name", sort=False)
    
    for col in cols:
        df[f"{col}_roll{window}_mean"] = g[col].rolling(window).mean().reset_index(level=0, drop=True)
        df[f"{col}_roll{window}_std"] = g[col].rolling(window).std().reset_index(level=0, drop=True)

    return df

train_df = add_rolling(train_df, window=3)
test_df = add_rolling(test_df, window=3)

In [8]:
print(train_df.shape)
train_df.head(1)

(7812, 62)


,Country Name,Year,Access to electricity (% of population),Agricultural land (% of land area),Agricultural land (sq. km),"Agriculture, forestry, and fishing, value added (% of GDP)","Annual freshwater withdrawals, total (% of internal resources)","Annual freshwater withdrawals, total (billion cubic meters)",Arable land (% of land area),Average precipitation in depth (mm per year),Carbon dioxide (CO2) emissions (total) excluding LULUCF (Mt CO2e),Cereal yield (kg per hectare),Electric power consumption (kWh per capita),Electricity production from coal sources (% of total),Electricity production from hydroelectric sources (% of total),Electricity production from natural gas sources (% of total),Electricity production from nuclear sources (% of total),Electricity production from oil sources (% of total),"Electricity production from renewable sources, excluding hydroelectric (% of total)","Electricity production from renewable sources, excluding hydroelectric (kWh)",Energy use (kg of oil equivalent per capita),"Energy use (kg of oil equivalent) per $1,000 GDP (constant 2021 PPP)","Foreign direct investment, net inflows (% of GDP)",Forest area (% of land area),Forest area (sq. km),"Mortality rate, under-5 (per 1,000 live births)",Population growth (annual %),Population in urban agglomerations of more than 1 million (% of total population),"Population, total","Primary completion rate, total (% of relevant age group)",Renewable electricity output (% of total electricity output),Renewable energy consumption (% of total final energy consumption),"School enrollment, primary and secondary (gross), gender parity index (GPI)",Urban population,Urban population (% of total population),Urban population growth (annual %),fossil_share,clean_share,energy_x_fossil,energy_x_clean,urban_energy,ag_land_sqkm_per_capita,forest_area_sqkm_per_capita,water_withdrawal_bcm_per_capita,"log1p_Population, total",log1p_Urban population,log1p_Energy use (kg of oil equivalent per capita),log1p_Electric power consumption (kWh per capita),log1p_Agricultural land (sq. km),log1p_Forest area (sq. km),"log1p_Electricity production from renewable sources, excluding hydroelectric (kWh)",Energy use (kg of oil equivalent per capita)_yoy_pct,Electric power consumption (kWh per capita)_yoy_pct,"Population, total_yoy_pct",Urban population_yoy_pct,fossil_share_yoy_pct,Energy use (kg of oil equivalent per capita)_roll3_mean,Energy use (kg of oil equivalent per capita)_roll3_std,Electric power consumption (kWh per capita)_roll3_mean,Electric power consumption (kWh per capita)_roll3_std,fossil_share_roll3_mean,fossil_share_roll3_std
0,Afghanistan,1990,4.4,58.322984,380400.0,38.627892,52.007095,24.521345,12.127624,327.0,2.9084,1200.6,0.0,0.0,93.537318,0.0,0.0,6.462682,0.0,0.0,0.0,0.0,0.004828,1.852782,12084.4,180.7,1.434588,12.86206,12045660.0,31.666389,93.537318,23.0,0.53706,2079567.0,17.264035,2.105964,6.462682,93.537318,0.0,0.0,0.0,0.03158,0.001003,0.000002,16.304215,14.547671,0.0,0.0,12.848981,9.399753,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
print(test_df.shape)
test_df.head(1)

(1008, 62)


,Country Name,Year,Access to electricity (% of population),Agricultural land (% of land area),Agricultural land (sq. km),"Agriculture, forestry, and fishing, value added (% of GDP)","Annual freshwater withdrawals, total (% of internal resources)","Annual freshwater withdrawals, total (billion cubic meters)",Arable land (% of land area),Average precipitation in depth (mm per year),Carbon dioxide (CO2) emissions (total) excluding LULUCF (Mt CO2e),Cereal yield (kg per hectare),Electric power consumption (kWh per capita),Electricity production from coal sources (% of total),Electricity production from hydroelectric sources (% of total),Electricity production from natural gas sources (% of total),Electricity production from nuclear sources (% of total),Electricity production from oil sources (% of total),"Electricity production from renewable sources, excluding hydroelectric (% of total)","Electricity production from renewable sources, excluding hydroelectric (kWh)",Energy use (kg of oil equivalent per capita),"Energy use (kg of oil equivalent) per $1,000 GDP (constant 2021 PPP)","Foreign direct investment, net inflows (% of GDP)",Forest area (% of land area),Forest area (sq. km),"Mortality rate, under-5 (per 1,000 live births)",Population growth (annual %),Population in urban agglomerations of more than 1 million (% of total population),"Population, total","Primary completion rate, total (% of relevant age group)",Renewable electricity output (% of total electricity output),Renewable energy consumption (% of total final energy consumption),"School enrollment, primary and secondary (gross), gender parity index (GPI)",Urban population,Urban population (% of total population),Urban population growth (annual %),fossil_share,clean_share,energy_x_fossil,energy_x_clean,urban_energy,ag_land_sqkm_per_capita,forest_area_sqkm_per_capita,water_withdrawal_bcm_per_capita,"log1p_Population, total",log1p_Urban population,log1p_Energy use (kg of oil equivalent per capita),log1p_Electric power consumption (kWh per capita),log1p_Agricultural land (sq. km),log1p_Forest area (sq. km),"log1p_Electricity production from renewable sources, excluding hydroelectric (kWh)",Energy use (kg of oil equivalent per capita)_yoy_pct,Electric power consumption (kWh per capita)_yoy_pct,"Population, total_yoy_pct",Urban population_yoy_pct,fossil_share_yoy_pct,Energy use (kg of oil equivalent per capita)_roll3_mean,Energy use (kg of oil equivalent per capita)_roll3_std,Electric power consumption (kWh per capita)_roll3_mean,Electric power consumption (kWh per capita)_roll3_std,fossil_share_roll3_mean,fossil_share_roll3_std
0,Afghanistan,2021,97.7,58.741548,383130.0,33.597619,43.015907,20.282,12.003434,327.0,12.4864,2099.1,0.0,0.0,71.656183,0.0,0.0,21.765525,6.578292,77910000.0,0.0,0.0,0.144467,1.852782,12084.4,59.3,2.356098,10.839313,40000412.0,0.0,78.234475,20.0,0.0,10139249.0,25.347862,2.694569,21.765525,78.234475,0.0,0.0,0.0,0.009578,0.000302,5.070448e-07,17.5044,16.131925,0.0,0.0,12.856132,9.399753,18.171065,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
train_df.to_csv(os.path.join('..', 'data', 'processed', 'feature_engineered_train.csv'), index=False)
test_df.to_csv(os.path.join('..', 'data', 'processed', 'feature_engineered_test.csv'), index=False)

print("✅ Feature engineering complete.")
print("Train shape:", train_df.shape)
print("Eval shape:", test_df.shape)

✅ Feature engineering complete.
Train shape: (7812, 62)
Eval shape: (1008, 62)
